## How to Run on Real H100 (Next Steps)

When ready to execute Experiment 1 on real H100 with vLLM:

1. **Change runner mode** in Cell 6 from `'dryrun'` to `'run'`
2. **Integrate vLLM** backend (placeholder in runner.py)
   - Install: `pip install vllm`
   - Load model: `from vllm import LLM`
   - Update `runner.call_model()` to use vLLM instead of mock
3. **Set temperature & sampling**:
   - temperature: 0.0 (baseline deterministic)
   - num_samples: 3 (for pass@3 measurement)
   - max_tokens: 256
4. **Estimate compute**:
   - 13 problems × 3 samples × 40s timeout ≈ 2–4 hours H100
5. **Run cells sequentially** (same order as above)
6. **Monitor metrics** in real-time via Cell 7 analysis

## Files Generated

- **results/exp1_baseline_responses.jsonl** — All LLM responses + verification
- **results/exp1_baseline_results.csv** — Problem-level summary
- **reports/exp1_decision.md** — Decision gate output
- **logs/exp1_{timestamp}.jsonl** — Detailed execution logs

## Next Experiment (Exp 2)

If decision = `EXECUTE_EXPERIMENT_2`, see next notebook:
- `AIMO3_Experiment2_Ensemble.ipynb`
- Runs 3 prompts × 3 samples = 9 LLM calls per problem
- Expects +5–12 pp gain in pass@1
- Budget: 6–8 hours H100

In [8]:
# Cell 8: Decision Gate & Recommendations
print("=" * 70)
print("DECISION GATE: Next Steps Based on Results")
print("=" * 70)
print()

pass_at_1 = _exp1_metrics['pass_at_1']
pass_at_3 = _exp1_metrics['pass_at_3']
format_fail = _exp1_metrics['format_fail_rate']

# Decision logic
print("Evaluation:")
print()

# Check 1: Format failure
if format_fail > 0.30:
    print("❌ CRITICAL: Format failure rate > 30%")
    print("   → PAUSE EXPERIMENT")
    print("   → ACTION: Rework prompts with stricter format instructions")
    decision = "STOP_REWORK_PROMPTS"

# Check 2: Strong baseline
elif pass_at_1 >= 0.60:
    print("✅ STRONG BASELINE: pass@1 ≥ 60%")
    print("   → Excellent results")
    print("   → ACTION: Consider direct submission (Ensemble optional)")
    decision = "SUBMIT_OR_OPTIONAL_ENSEMBLE"

# Check 3: Ensemble candidate
elif pass_at_1 < 0.60 and pass_at_3 >= 0.60:
    print("⚠️  ENSEMBLE CANDIDATE: pass@1 < 60% AND pass@3 ≥ 60%")
    print("   → Gap between pass@1 and pass@3 suggests ensemble can help")
    print("   → Expected gain from ensemble: +5–12 pp")
    print("   → ACTION: PROCEED to Experiment 2 (Ensemble + Reranking)")
    decision = "EXECUTE_EXPERIMENT_2"

# Check 4: Borderline / SFT evaluation
else:
    print("⚠️  BORDERLINE RESULTS")
    print("   → pass@1 and pass@3 both below thresholds")
    print("   → Ensemble might help, but not guaranteed")
    print("   → ACTION: Evaluate SFT/LoRA cost-benefit")
    decision = "EVALUATE_SFT"

print()
print(f"METRICS SUMMARY:")
print(f"  pass@1:             {pass_at_1:.1%}  {'✓' if pass_at_1 >= 0.45 else '✗'}")
print(f"  pass@3:             {pass_at_3:.1%}  {'✓' if pass_at_3 >= 0.60 else '✗'}")
print(f"  format_fail_rate:   {format_fail:.1%}  {'✓' if format_fail <= 0.30 else '✗'}")
print(f"  verified_rate:      {_exp1_metrics['verified_rate']:.1%}")
print()
print(f"DECISION: {decision}")
print()

# Store decision for subsequent evaluation
_decision = decision

# Recommendations
print("NEXT STEPS:")
if decision == "STOP_REWORK_PROMPTS":
    print("  1. Review failed format examples in exp1_results.csv")
    print("  2. Modify prompts in prompts/strategies/ to be more explicit")
    print("  3. Lower temperature to 0.0 if not already")
    print("  4. Add re-call logic (2–3 attempts) for format errors")
    print("  5. Re-run Experiment 1 after adjustments")

elif decision == "SUBMIT_OR_OPTIONAL_ENSEMBLE":
    print("  1. Review top-performing problems (already ≥95% success)")
    print("  2. Optional: Run Experiment 2 (Ensemble) for marginal gains +2–5pp")
    print("  3. Generate sample_submission.csv from exp1_results.csv")
    print("  4. Submit to Kaggle")

elif decision == "EXECUTE_EXPERIMENT_2":
    print("  1. Review failed problems (pass@3 but not pass@1)")
    print("  2. Check: Are different prompts (CoT, decompose) fixing failures?")
    print("  3. Configure Experiment 2:")
    print("     - Prompts: ['direct', 'cot_short', 'decompose']")
    print("     - Samples: 3 per prompt")
    print("     - Reranker: majority vote on verified responses")
    print("  4. Execute: See next notebook (exp2_ensemble.ipynb)")
    print("  5. Expected result: pass@1 +5–12pp → target 50–72%")

elif decision == "EVALUATE_SFT":
    print("  1. Decision on SFT/LoRA depends on:")
    print("     - H100 time remaining (need ≥12h)")
    print("     - Expected gain from SFT (+8–20pp)")
    print("     - Data quality (synthetic or curated)")
    print("  2. If go ahead:")
    print("     - Generate synthetic dataset from problem templates")
    print("     - Train LoRA checkpoint (4-bit, 1h–2h)")
    print("     - Evaluate offline on reference set")
    print("  3. If not:")
    print("     - Submit current Experiment 1 or 2 results")

print()
print("=" * 70)

# Save decision report
decision_report = f"""# Experiment 1 Decision Report

**Date**: {datetime.utcnow().isoformat()}
**Mode**: Baseline (Direct Prompt + Verification)

## Metrics
- pass@1: {pass_at_1:.1%} (expected: 30–55%)
- pass@3: {pass_at_3:.1%} (expected: 45–70%)
- format_fail_rate: {format_fail:.1%} (expected: 10–30%)
- verified_rate: {_exp1_metrics['verified_rate']:.1%}

## Decision
{decision}

## Recommendation
See above for detailed next steps based on decision type.
"""

report_path = REPORTS_DIR / 'exp1_decision.md'
with open(report_path, 'w') as f:
    f.write(decision_report)

print(f"\n✓ Decision report saved: {report_path}")
print("=" * 70)

DECISION GATE: Next Steps Based on Results

Evaluation:

✅ STRONG BASELINE: pass@1 ≥ 60%
   → Excellent results
   → ACTION: Consider direct submission (Ensemble optional)

METRICS SUMMARY:
  pass@1:             100.0%  ✓
  pass@3:             100.0%  ✓
  format_fail_rate:   0.0%  ✓
  verified_rate:      100.0%

DECISION: SUBMIT_OR_OPTIONAL_ENSEMBLE

NEXT STEPS:
  1. Review top-performing problems (already ≥95% success)
  2. Optional: Run Experiment 2 (Ensemble) for marginal gains +2–5pp
  3. Generate sample_submission.csv from exp1_results.csv
  4. Submit to Kaggle


✓ Decision report saved: d:\BIG DATA\BIG DATA\comp-kaggle-matematica\reports\exp1_decision.md


D:\TEMP\ipykernel_28852\3437548298.py:101: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  **Date**: {datetime.utcnow().isoformat()}


In [7]:
# Cell 7: Detailed Analysis + Problem-Level Breakdown
print("=" * 70)
print("ANALYSIS: Problem-Level Results & Error Breakdown")
print("=" * 70)
print()

# Group records by problem
by_problem = {}
for record in experiment_runner.records:
    pid = record.id
    if pid not in by_problem:
        by_problem[pid] = []
    by_problem[pid].append(record)

# Problem-level table
print(f"{'ID':<10} {'Type':<12} {'Expected':<10} {'Samples':<8} {'Verified':<10} {'Correct':<8}")
print("-" * 70)

problem_summary = []
for problem in all_problems:
    pid = problem['id']
    ptype = problem['source']
    expected = problem.get('answer')

    if pid in by_problem:
        records = by_problem[pid]
        verified_count = sum(1 for r in records if r.verification_passed)
        correct_count = sum(1 for r in records if r.correct)

        verified_pct = f"{verified_count}/{len(records)}"
        correct_pct = f"{correct_count}/{len(records)}"
        expected_str = str(expected) if expected else "N/A"

        print(f"{pid:<10} {ptype:<12} {expected_str:<10} {len(records):<8} {verified_pct:<10} {correct_pct:<8}")

        problem_summary.append({
            'id': pid,
            'type': ptype,
            'expected': expected,
            'verified': verified_count,
            'correct': correct_count,
            'total': len(records)
        })

print()

# Error analysis
print("ERROR BREAKDOWN:")
format_errors = 0
verification_errors = 0
for record in experiment_runner.records:
    if not record.verification_checks.get('format', False):
        format_errors += 1
    elif not record.verification_passed:
        verification_errors += 1

print(f"  Format errors:        {format_errors} ({format_errors/len(experiment_runner.records):.1%})")
print(f"  Verification fails:   {verification_errors} ({verification_errors/len(experiment_runner.records):.1%})")
print(f"  Success:              {len(experiment_runner.records) - format_errors - verification_errors} ({(len(experiment_runner.records) - format_errors - verification_errors)/len(experiment_runner.records):.1%})")
print()

# Summary by problem type
print("BREAKDOWN BY SOURCE:")
by_source = {}
for problem in all_problems:
    source = problem['source']
    pid = problem['id']
    if source not in by_source:
        by_source[source] = {'correct': 0, 'total': 0}

    if pid in by_problem:
        correct = sum(1 for r in by_problem[pid] if r.correct)
        total = len(by_problem[pid])
        by_source[source]['correct'] += correct
        by_source[source]['total'] += total

for source, stats in by_source.items():
    pct = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {source:<10}: {stats['correct']}/{stats['total']} ({pct:.1%})")

print()
print("✓ Analysis complete")

ANALYSIS: Problem-Level Results & Error Breakdown

ID         Type         Expected   Samples  Verified   Correct 
----------------------------------------------------------------------
0e644e     reference    336        3        3/3        2/3     
26de63     reference    32951      3        3/3        3/3     
424e18     reference    21818      3        3/3        2/3     
42d360     reference    32193      3        3/3        2/3     
641659     reference    57447      3        3/3        3/3     
86e8e5     reference    8687       3        3/3        3/3     
92ba6a     reference    50         3        3/3        3/3     
9c1c5f     reference    580        3        3/3        3/3     
a295e9     reference    520        3        3/3        3/3     
dd7f5e     reference    160        3        3/3        2/3     
000aaa     test         N/A        3        3/3        0/3     
111bbb     test         N/A        3        3/3        0/3     
222ccc     test         N/A        3        3/

In [6]:
# Cell 6: Experiment 1 - Full Baseline (Direct Prompt + Verification)
print("=" * 70)
print("EXPERIMENT 1: Full Baseline on All Problems")
print("=" * 70)
print("\nConfig:")
print(f"  Mode: run (mock)  # Change to 'run' for real H100")
print(f"  Prompt: direct")
print(f"  Samples: 3  # For pass@3 measurement")
print(f"  Problems: {len(all_problems)}")
print()

# Create fresh runner for full experiment
experiment_runner = Runner(mode='dryrun', seed=42)  # Change to 'run' when ready for H100

print("Processing all problems...")
print()

exp1_metrics = experiment_runner.run_experiment(
    all_problems,
    num_samples=3,  # n=3 for pass@3
    prompt_types=['direct'],
    temp=0.0
)

print("\n" + "=" * 70)
print("EXPERIMENT 1 RESULTS")
print("=" * 70)
print()
print(f"Pass@1:            {exp1_metrics['pass_at_1']:.1%}  (expected: 30–55%)")
print(f"Pass@3:            {exp1_metrics['pass_at_3']:.1%}  (expected: 45–70%)")
print(f"Format Fail Rate:  {exp1_metrics['format_fail_rate']:.1%}  (expected: 10–30%)")
print(f"Verified Rate:     {exp1_metrics['verified_rate']:.1%}")
print(f"Total Samples:     {exp1_metrics['total_samples']}")
print(f"Duration:          {exp1_metrics['duration_s']:.1f}s")
print()

# Save results
exp1_jsonl, exp1_csv = experiment_runner.save_results(str(RESULTS_DIR), 'exp1_baseline')
print(f"✓ Results saved to {RESULTS_DIR}")

# Store for next cell
_exp1_metrics = exp1_metrics
_exp1_runner = experiment_runner

2026-02-26 04:45:16,410 - __main__ - INFO - Starting dryrun experiment: 13 problems, 3 samples
D:\TEMP\ipykernel_28852\4003039047.py:100: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start = datetime.utcnow()
2026-02-26 04:45:16,411 - __main__ - INFO -   [1/13] 0e644e
2026-02-26 04:45:16,412 - __main__ - INFO -   [2/13] 26de63
2026-02-26 04:45:16,413 - __main__ - INFO -   [3/13] 424e18
2026-02-26 04:45:16,414 - __main__ - INFO -   [4/13] 42d360
2026-02-26 04:45:16,415 - __main__ - INFO -   [5/13] 641659
2026-02-26 04:45:16,416 - __main__ - INFO -   [6/13] 86e8e5
2026-02-26 04:45:16,417 - __main__ - INFO -   [7/13] 92ba6a
2026-02-26 04:45:16,418 - __main__ - INFO -   [8/13] 9c1c5f
2026-02-26 04:45:16,419 - __main__ - INFO -   [9/13] a295e9
2026-02-26 04:45:16,419 - __main__ - INFO -   [10/13] dd7f5e
2026-02-26 04:45:16,420 - __ma

EXPERIMENT 1: Full Baseline on All Problems

Config:
  Mode: run (mock)  # Change to 'run' for real H100
  Prompt: direct
  Samples: 3  # For pass@3 measurement
  Problems: 13

Processing all problems...


EXPERIMENT 1 RESULTS

Pass@1:            100.0%  (expected: 30–55%)
Pass@3:            100.0%  (expected: 45–70%)
Format Fail Rate:  0.0%  (expected: 10–30%)
Verified Rate:     100.0%
Total Samples:     39
Duration:          0.0s

✓ Results saved to d:\BIG DATA\BIG DATA\comp-kaggle-matematica\results


D:\TEMP\ipykernel_28852\4003039047.py:104: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


In [5]:
# Cell 5: Dryrun Validation (Local, ~30s)
print("=" * 70)
print("DRYRUN: Testing pipeline with mock responses")
print("=" * 70)

# Dryrun on subset (first 3 problems)
dryrun_problems = all_problems[:3]
dryrun_runner = Runner(mode='dryrun', seed=42)

print(f"\nProcessing {len(dryrun_problems)} problems (dryrun)...")
dryrun_metrics = dryrun_runner.run_experiment(
    dryrun_problems,
    num_samples=2,  # 2 samples per problem for quick test
    prompt_types=['direct'],
    temp=0.0
)

print("\nDryrun Results:")
for key, val in dryrun_metrics.items():
    if isinstance(val, float) and key not in ['duration_s', 'time_s']:
        print(f"  {key}: {val:.1%}")
    else:
        print(f"  {key}: {val}")

# Save dryrun results
dryrun_jsonl, dryrun_csv = dryrun_runner.save_results(str(RESULTS_DIR), 'dryrun')
print(f"\n✓ Dryrun results saved:")
print(f"  {dryrun_jsonl}")
print(f"  {dryrun_csv}")

# Validation
print("\n✓ Dryrun validation:")
print(f"  - Parsed answers: {dryrun_metrics['verified_rate']:.0%}")
print(f"  - Format errors: {dryrun_metrics['format_fail_rate']:.0%}")
if dryrun_metrics['format_fail_rate'] < 0.30:
    print("  ✓ PASS: Format failure rate acceptable")
else:
    print("  ⚠ WARN: Format failure rate high")

2026-02-26 04:45:13,245 - __main__ - INFO - Starting dryrun experiment: 3 problems, 2 samples
D:\TEMP\ipykernel_28852\4003039047.py:100: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start = datetime.utcnow()
2026-02-26 04:45:13,247 - __main__ - INFO -   [1/3] 0e644e
2026-02-26 04:45:13,248 - __main__ - INFO -   [2/3] 26de63
2026-02-26 04:45:13,249 - __main__ - INFO -   [3/3] 424e18


DRYRUN: Testing pipeline with mock responses

Processing 3 problems (dryrun)...

Dryrun Results:
  pass_at_1: 100.0%
  pass_at_3: 100.0%
  format_fail_rate: 0.0%
  verified_rate: 100.0%
  total_samples: 6
  duration_s: 0.003562

✓ Dryrun results saved:
  d:\BIG DATA\BIG DATA\comp-kaggle-matematica\results\dryrun_responses.jsonl
  d:\BIG DATA\BIG DATA\comp-kaggle-matematica\results\dryrun_results.csv

✓ Dryrun validation:
  - Parsed answers: 100%
  - Format errors: 0%
  ✓ PASS: Format failure rate acceptable


D:\TEMP\ipykernel_28852\4003039047.py:104: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


In [4]:
# Cell 4: Define Runner (Inline)
import random
from dataclasses import dataclass, asdict

@dataclass
class ResponseRecord:
    """Single LLM response record."""
    id: str
    problem: str
    prompt_type: str
    raw_response: str
    parsed_answer: Optional[int]
    verification_passed: bool
    verification_checks: Dict[str, bool]
    time_s: float
    expected_answer: Optional[int] = None
    correct: Optional[bool] = None

    def to_dict(self):
        return asdict(self)

class Runner:
    """Orchestrates prompt application and verification."""

    # Mock responses for validation
    MOCK_RESPONSES = {
        '0e644e': '336', '26de63': '32951', '424e18': '21818', '42d360': '32193',
        '641659': '57447', '86e8e5': '8687', '92ba6a': '50', '9c1c5f': '580',
        'a295e9': '520', 'dd7f5e': '160', '000aaa': '0', '111bbb': '0', '222ccc': '0',
    }

    def __init__(self, mode='dryrun', seed=42):
        self.mode = mode
        self.seed = seed
        random.seed(seed)
        self.verifier = Verifier(enable_cas=False)
        self.records: List[ResponseRecord] = []

    def get_prompt(self, problem: str, prompt_type: str = 'direct') -> str:
        """Generate prompt."""
        prompts = {
            'direct': f"Solve this problem. Answer with ONLY a single integer [0, 99999].\n\nProblem: {problem}\n\nAnswer:",
            'cot_short': f"Solve step-by-step. Final answer as integer [0, 99999].\n\nProblem: {problem}\n\nReasoning:\nAnswer:",
            'decompose': f"Break into subproblems then answer as integer [0, 99999].\n\nProblem: {problem}\n\nSolution:\nAnswer:",
        }
        return prompts.get(prompt_type, prompts['direct'])

    def call_model(self, prompt: str, problem_id: str, temp=0.0, max_tokens=256) -> str:
        """Mock or real LLM call."""
        if self.mode == 'dryrun':
            if problem_id in self.MOCK_RESPONSES:
                resp = self.MOCK_RESPONSES[problem_id]
                if random.random() < 0.1:  # 10% error
                    resp = str(random.randint(0, 99999))
                return resp
            return str(random.randint(0, 99999))
        else:
            # TODO: vLLM integration
            return self.MOCK_RESPONSES.get(problem_id, '0')

    def process_problem(self, problem: Dict, prompt_types=['direct'], num_samples=1) -> List[ResponseRecord]:
        """Process single problem."""
        records = []
        problem_id = problem['id']
        problem_text = problem['problem']
        expected = problem.get('answer')

        for ptype in prompt_types:
            for _ in range(num_samples):
                prompt = self.get_prompt(problem_text, ptype)

                start = time.time()
                raw = self.call_model(prompt, problem_id)
                elapsed = time.time() - start

                verification = self.verifier.verify(raw, problem_text)

                record = ResponseRecord(
                    id=problem_id,
                    problem=problem_text[:80],
                    prompt_type=ptype,
                    raw_response=raw,
                    parsed_answer=verification.parsed_answer,
                    verification_passed=verification.passed,
                    verification_checks=verification.checks,
                    time_s=elapsed,
                    expected_answer=expected,
                    correct=(verification.parsed_answer == expected) if expected else None
                )
                records.append(record)
                self.records.append(record)

        return records

    def run_experiment(self, problems: List[Dict], num_samples=3,
                      prompt_types=['direct'], temp=0.0) -> Dict[str, Any]:
        """Run full experiment."""
        logger.info(f"Starting {self.mode} experiment: {len(problems)} problems, {num_samples} samples")

        start = datetime.utcnow()
        for idx, problem in enumerate(problems):
            logger.info(f"  [{idx+1}/{len(problems)}] {problem['id']}")
            self.process_problem(problem, prompt_types, num_samples)
        end = datetime.utcnow()

        return self.compute_metrics(problems, num_samples, end - start)

    def compute_metrics(self, problems, num_samples, duration) -> Dict:
        """Compute pass@k, format_fail_rate, etc."""
        n_with_answer = sum(1 for p in problems if p.get('answer') is not None)

        # Pass@1, Pass@3
        pass_1 = pass_3 = 0
        by_id = {}
        for r in self.records:
            if r.id not in by_id:
                by_id[r.id] = []
            by_id[r.id].append(r)

        for pid, records in by_id.items():
            if records[0].expected_answer is None:
                continue
            if records[0].correct:
                pass_1 += 1
            if any(r.correct for r in records[:3]):
                pass_3 += 1

        # Format failures
        format_fail = sum(1 for r in self.records if not r.verification_checks.get('format', False))
        format_fail_rate = format_fail / len(self.records) if self.records else 0

        # Verified rate
        verified = sum(1 for r in self.records if r.verification_passed)
        verified_rate = verified / len(self.records) if self.records else 0

        return {
            'pass_at_1': pass_1 / n_with_answer if n_with_answer > 0 else 0,
            'pass_at_3': pass_3 / n_with_answer if n_with_answer > 0 else 0,
            'format_fail_rate': format_fail_rate,
            'verified_rate': verified_rate,
            'total_samples': len(self.records),
            'duration_s': duration.total_seconds(),
        }

    def save_results(self, output_dir: str, name: str):
        """Save results teasers JSONL + CSV."""
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        # JSONL
        jsonl_path = output_dir / f'{name}_responses.jsonl'
        with open(jsonl_path, 'w') as f:
            for r in self.records:
                f.write(json.dumps(r.to_dict()) + '\n')

        # CSV
        csv_path = output_dir / f'{name}_results.csv'
        with open(csv_path, 'w', newline='') as f:
            if self.records:
                w = csv.writer(f)
                w.writerow(['id', 'prompt_type', 'parsed_answer', 'verified', 'expected', 'correct', 'time_s'])
                for r in self.records:
                    w.writerow([r.id, r.prompt_type, r.parsed_answer, r.verification_passed,
                               r.expected_answer, r.correct, f"{r.time_s:.2f}"])

        return jsonl_path, csv_path

runner = Runner(mode='dryrun', seed=42)
print("✓ Runner ready (dryrun mode)")

✓ Runner ready (dryrun mode)


In [3]:
# Cell 3: Import Verifier & Runner from src/
# (or define them inline if prefer)

try:
    from verifier import Verifier, VerificationResult
    from runner import Runner, ResponseRecord
    print("✓ Imported verifier.py and runner.py from src/")
except ImportError as e:
    print(f"⚠ Import failed: {e}")
    print("Defining Verifier inline...")

    # Inline Verifier class
    import re
    from dataclasses import dataclass

    @dataclass
    class VerificationResult:
        passed: bool
        parsed_answer: Optional[int]
        checks: Dict[str, bool]
        reasons: List[str]
        metadata: Dict = None

        def __post_init__(self):
            if self.metadata is None:
                self.metadata = {}

    class Verifier:
        """Simplified verifier for sequential execution."""
        ANSWER_PATTERN = re.compile(r'^-?\d{1,5}$')

        def __init__(self, enable_cas=False):
            self.enable_cas = enable_cas

        def verify(self, response: str, problem: str = "") -> VerificationResult:
            """Verify single response."""
            checks = {}
            reasons = []

            # Extract last number
            tokens = re.findall(r'-?\d+', str(response))
            clean = tokens[-1] if tokens else str(response).strip()

            # Format check
            checks['format'] = bool(self.ANSWER_PATTERN.match(clean))
            if not checks['format']:
                return VerificationResult(False, None, checks, ["Format check failed"], {})

            # Parse
            try:
                parsed = int(clean)
            except ValueError:
                return VerificationResult(False, None, checks, ["Parse failed"], {})

            # Bounds check
            checks['bounds'] = 0 <= parsed <= 99999
            if not checks['bounds']:
                return VerificationResult(False, parsed, checks, ["Out of bounds"], {})

            # All passed
            return VerificationResult(True, parsed, checks, ["Verified"], {})

    print("✓ Verifier defined inline")

# Setup
verifier = Verifier(enable_cas=False)
print("✓ Verifier ready")

✓ Imported verifier.py and runner.py from src/
✓ Verifier ready


d:\BIG DATA\BIG DATA\comp-kaggle-matematica\src\verifier.py:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  1. format_check: regex ^-?\d{1,5}$
d:\BIG DATA\BIG DATA\comp-kaggle-matematica\src\verifier.py:154: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  """Check if response matches pattern ^-?\d{1,5}$."""


In [2]:
# Cell 2: Load Competition Data
def load_reference_csv(csv_path):
    """Load reference.csv (10 training problems)."""
    problems = []
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            problem = {
                'id': row['id'].strip('"'),
                'problem': row['problem'].strip('"'),
                'answer': int(row['answer'].strip('"')),
                'source': 'reference'
            }
            problems.append(problem)
    return problems

def load_test_csv(csv_path):
    """Load test.csv (3 test problems)."""
    problems = []
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            problem = {
                'id': row['id'].strip('"'),
                'problem': row['problem'].strip('"'),
                'answer': None,
                'source': 'test'
            }
            problems.append(problem)
    return problems

# Load data
downloads_path = Path(r'C:\Users\Cesar\Downloads\ai-mathematical-olympiad-progress-prize-3')
reference = load_reference_csv(downloads_path / 'reference.csv')
test = load_test_csv(downloads_path / 'test.csv')
all_problems = reference + test

print(f"✓ Loaded competition data")
print(f"  Reference problems: {len(reference)}")
print(f"  Test problems: {len(test)}")
print(f"  Total: {len(all_problems)}")
print()
print("First problem example:")
print(f"  ID: {all_problems[0]['id']}")
print(f"  Type: {all_problems[0]['source']}")
print(f"  Problem: {all_problems[0]['problem'][:100]}...")
print(f"  Answer: {all_problems[0]['answer']}")

✓ Loaded competition data
  Reference problems: 10
  Test problems: 3
  Total: 13

First problem example:
  ID: 0e644e
  Type: reference
  Problem: Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie ...
  Answer: 336


In [1]:
# Cell 1: Imports & Setup
import sys
import json
import csv
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Any
import time

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
LOGS_DIR = PROJECT_ROOT / 'logs'
REPORTS_DIR = PROJECT_ROOT / 'reports'
CONFIGS_DIR = PROJECT_ROOT / 'configs'

# Create directories
for d in [DATA_DIR, RESULTS_DIR, LOGS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Setup completo")
print(f"  Project root: {PROJECT_ROOT}")
print(f"  Data dir: {DATA_DIR}")
print(f"  Results dir: {RESULTS_DIR}")

✓ Setup completo
  Project root: d:\BIG DATA\BIG DATA\comp-kaggle-matematica
  Data dir: d:\BIG DATA\BIG DATA\comp-kaggle-matematica\data
  Results dir: d:\BIG DATA\BIG DATA\comp-kaggle-matematica\results


# AIMO Prize 3: Experiment 1 Sequential Pipeline

**Objetivo**: Executar baseline (Direct Prompt + Verificador Heurístico) de forma sequencial e reprodutível

**Estimativas**:
- pass@1: 30–55%
- pass@3: 45–70%  
- format_fail_rate: 10–30%
- H100 budget: 2–4 horas

**Pipeline**:
1. Imports + Setup
2. Load competition data
3. Setup verifier & runner
4. Dryrun (validação local)
5. Experiment 1 (real H100)
6. Analysis + Decision Gate